### ingest_daily_support_tickets

In [1]:
import os
import pandas as pd
import boto3
from io import StringIO
from sqlalchemy import create_engine
from datetime import datetime, timedelta
import time

from dotenv import load_dotenv

load_dotenv()

True

In [2]:
# ---------- CONFIG ----------
db_config = {
    "host": "localhost",
    "port": "3306",
    "user": "root",  # change
    "password": "root", # change
    "database": "careplus_support_db"
}

S3_BUCKET = "care-plus-databucket" 
S3_PREFIX = "support-tickets/raw/"  
DATE_TRACKER_FILE = "date_tracker.txt"

import os

AWS_CONFIG = {
    "aws_access_key_id": os.getenv("AWS_ACCESS_KEY"),
    "aws_secret_access_key": os.getenv("SECRET_KEY"),
    "region_name": os.getenv("REGION")
}

In [3]:
# ---------- UTILITY FUNCTIONS ----------
def get_engine(config):
    return create_engine(f"mysql+pymysql://{config['user']}:{config['password']}@{config['host']}:{config['port']}/{config['database']}")

def upload_to_s3(df, bucket, key):
    csv_buffer = StringIO()
    df.to_csv(csv_buffer, index=False)

    s3 = boto3.client('s3', **AWS_CONFIG)
    s3.put_object(Bucket=bucket, Key=key, Body=csv_buffer.getvalue())
    print(f"✅ Uploaded to s3://{bucket}/{key}")

def read_last_date(file_path):
    if os.path.exists(file_path):
        with open(file_path, 'r') as f:
            return f.read().strip()
    return "2025-06-30"  # Starting point before 1st July

def update_last_date(file_path, new_date):
    with open(file_path, 'w') as f:
        f.write(new_date)

def get_next_date(last_date_str):
    last_date = datetime.strptime(last_date_str, "%Y-%m-%d")
    next_date = last_date + timedelta(days=1)
    return next_date.strftime("%Y-%m-%d")

# ---------- MAIN INGESTION LOGIC ----------
def run_ingestion():
    engine = get_engine(db_config)
    last_date = read_last_date(DATE_TRACKER_FILE)
    next_date = get_next_date(last_date)

    # Query only that day’s data
    query = f"""
        SELECT * FROM support_tickets
        WHERE DATE(created_at) = '{next_date}';
    """
    df = pd.read_sql(query, engine)
    print(df.shape)
    print(df.head())

    if df.empty:
        print(f"⚠️ No data found for {next_date}. Skipping upload.")
        return

    # Upload to S3
    s3_key = f"{S3_PREFIX}support_tickets_{next_date}.csv"
    upload_to_s3(df, S3_BUCKET, s3_key)

    # Update date tracker
    update_last_date(DATE_TRACKER_FILE, next_date)
    print(f"📅 Updated tracker to {next_date}")



In [ ]:
count = 0
if __name__ == "__main__":
    while True:
        count += 1
        run_ingestion()
        print(f"Total runs:  {count}")
        time.sleep(180)   # 3 minutes = 180 seconds

(23, 10)
    ticket_id        created_at       resolved_at   agent priority  \
0  TCK0724000  2025-07-24 01:07  2025-07-24 07:50   Kavya   Medium   
1  TCK0724001  2025-07-24 01:19  2025-07-24 09:09  Ananya   Medium   
2  TCK0724002  2025-07-24 02:52               NaN   Sneha     High   
3  TCK0724003  2025-07-24 03:16  2025-07-24 13:23   Sneha   Medium   
4  TCK0724004  2025-07-24 03:17  2025-07-24 08:46   Arjun    Medum   

  num_interactions         IssUeCat   channel    status agent_feedback  
0                3  Payment Failure     Email  Resolved                 
1                1  Payment Failure     Email  Resolved                 
2                8       Bug Report     Phone      Open                 
3                3       Bug Report  Web Form  Resolved                 
4          -999999       Bug Report  Web Form  Resolved                 
✅ Uploaded to s3://care-plus-databucket/support-tickets/raw/support_tickets_2025-07-24.csv
📅 Updated tracker to 2025-07-24
Total run